In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, DateType, StringType
bronze_orders_new = spark.read.table("bronze_orders_new")
silver_orders_new = (
    bronze_orders_new
    .select(
        F.trim(F.col('customer_name')).alias('customer_name'),
        F.trim(F.upper(F.col('product'))).alias('product'),
        F.col('order_id').cast(IntegerType()).alias('order_id'),
        F.col('customer_id').cast(StringType()).alias('customer_id'),
        F.col('amount').cast(DoubleType()).alias('amount'),
        F.to_date(F.col('order_date')).alias('order_date'),
        F.col('ingestion_timestamp')
    )
    .filter(
        F.col('order_id').isNotNull()&
        F.col('customer_id').isNotNull()&
        (F.col('customer_name') != '')&
        F.col('amount').isNotNull()
    )
)
display(silver_orders_new)

(
    silver_orders_new
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('default.silver_orders_new')
)

customer_name,product,order_id,customer_id,amount,order_date,ingestion_timestamp
John Doe,LAPTOP,101,C101,1500.5,2026-10-01,2026-10-05T18:26:35.304Z
Jane Smith,PHONE,102,C102,750.0,2026-10-01,2026-10-05T18:26:35.304Z
John Doe,HEADPHONES,105,C101,50.0,2026-10-03,2026-10-05T18:26:35.304Z
John Doe,LAPTOP,106,C105,1200.0,2026-10-03,2026-10-05T18:26:35.304Z
Jane Smith,MONITOR,107,C102,300.0,2026-10-04,2026-10-05T18:26:35.304Z
Alice Johnson,KEYBOARD,108,C106,85.5,2026-10-04,2026-10-05T18:26:35.304Z
Bob Brown,MOUSE,109,C107,25.0,2026-10-05,2026-10-05T18:26:35.304Z
Charlie Davis,null,110,C108,150.0,2026-10-05,2026-10-05T18:26:35.304Z
Alice Johnson,DESK LAMP,111,C106,45.0,2026-10-05,2026-10-05T18:26:35.304Z
David Miller,GAMING CHAIR,112,C109,350.0,2026-10-06,2026-10-05T18:26:35.304Z
